# 作业 02：把多个输入接入上周的训练模板

作业 01 比较了同一个样本的两种表示：两个标量输入张量，或一个包含两个数的输入张量。本题把多个样本分别放入 x1、x2，练习补全训练代码。

请参考上周模板，让同一个预测函数同时使用 x1 和 x2。所有样本共享 w1、w2、b；每次预测都需要两个输入。下面提供函数和数据，参数创建、损失、梯度与更新由你完成。

本文件可以独立运行，不依赖作业 01 的内核变量。


In [1]:
import torch

def predict(x1, x2, w1, w2, b):
    return w1 * x1 + w2 * x2 + b


## 训练数据：每一行是同一个样本的两个输入和一个目标

下面 x1、x2、y 的 shape 都为 `(6,)`，表示 6 个样本。`predict(x1, x2, ...)` 同时使用两个输入张量，返回 6 个预测值。

| 样本 i | x1[i] | x2[i] | y[i] |
|---|---|---|---|
| 0 | -2 | 1 | -6 |
| 1 | -1 | 0 | -1 |
| 2 | 0 | 2 | -5 |
| 3 | 1 | -1 | 6 |
| 4 | 2 | 3 | -4 |
| 5 | 3 | 1 | 4 |

训练时不要修改输入和目标，也不要直接填写猜测的参数答案。


In [2]:
x1 = torch.tensor([-2., -1., 0., 1., 2., 3.])
x2 = torch.tensor([1., 0., 2., -1., 3., 1.])
y = torch.tensor([-6., -1., -5., 6., -4., 4.])
print('x1:', x1, 'shape:', tuple(x1.shape))
print('x2:', x2, 'shape:', tuple(x2.shape))
print('y: ', y, 'shape:', tuple(y.shape))


x1: tensor([-2., -1.,  0.,  1.,  2.,  3.]) shape: (6,)
x2: tensor([ 1.,  0.,  2., -1.,  3.,  1.]) shape: (6,)
y:  tensor([-6., -1., -5.,  6., -4.,  4.]) shape: (6,)


## 你的训练区：修改上周模板

创建三个浮点标量参数 w1、w2、b，都启用梯度，并一起交给优化器。每轮调用给定的 predict 函数，同时传入 x1、x2，得到 shape 为 `(6,)` 的 y_pred。

使用绝对值损失：先得到 `(6,)` 的逐样本误差，再用 `.mean()` 得到 shape 为 `()` 的平均损失。只对这个平均损失 backward 一次，然后更新全部参数。

可以从三个参数都为 0、2000 轮、学习率 `0.1 / (epoch ** 0.5)` 开始。这里预测公式是线性的，零初始化与上周的平方参数例子不同。

补完后删除 NotImplementedError。


In [3]:
# 创建三个浮点标量参数，都启用梯度
w1 = torch.tensor(0.0, requires_grad=True)
w2 = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

# 三个参数一起交给 SGD 优化器
optimizer = torch.optim.SGD([w1, w2, b], lr=0.1)

for epoch in range(1, 2001):
    # 设置本轮学习率：0.1 / sqrt(epoch)
    lr = 0.1 / (epoch ** 0.5)
    for g in optimizer.param_groups:
        g['lr'] = lr
    # 清空旧梯度
    optimizer.zero_grad()
    # 调用 predict，同时使用 x1 和 x2，得到 shape 为 (6,) 的预测
    y_pred = predict(x1, x2, w1, w2, b)
    # 逐样本绝对误差 (6,)，再取平均得到标量损失 ()
    errors = torch.abs(y - y_pred)
    loss = errors.mean()
    # 反向传播并更新全部参数
    loss.backward()
    optimizer.step()

# 在 no_grad 中用训练后的参数重新计算预测与损失
with torch.no_grad():
    y_pred_final = predict(x1, x2, w1, w2, b)
    errors_final = torch.abs(y - y_pred_final)
    loss_final = errors_final.mean()

# 打印三个参数、输入、目标、预测、逐样本损失、平均损失及其 shape
print('w1 =', w1.item(), 'shape =', tuple(w1.shape))
print('w2 =', w2.item(), 'shape =', tuple(w2.shape))
print('b  =', b.item(), 'shape =', tuple(b.shape))
print('x1 =', x1, 'shape =', tuple(x1.shape))
print('x2 =', x2, 'shape =', tuple(x2.shape))
print('y  =', y, 'shape =', tuple(y.shape))
print('y_pred =', y_pred_final, 'shape =', tuple(y_pred_final.shape))
print('逐样本损失 =', errors_final, 'shape =', tuple(errors_final.shape))
print('平均损失 =', loss_final.item(), 'shape =', tuple(loss_final.shape))


w1 = 1.999006986618042 shape = ()
w2 = -3.0010085105895996 shape = ()
b  = 0.9993674159049988 shape = ()
x1 = tensor([-2., -1.,  0.,  1.,  2.,  3.]) shape = (6,)
x2 = tensor([ 1.,  0.,  2., -1.,  3.,  1.]) shape = (6,)
y  = tensor([-6., -1., -5.,  6., -4.,  4.]) shape = (6,)
y_pred = tensor([-5.9997, -0.9996, -5.0026,  5.9994, -4.0056,  3.9954]) shape = (6,)
逐样本损失 = tensor([0.0003, 0.0004, 0.0026, 0.0006, 0.0056, 0.0046]) shape = (6,)
平均损失 = 0.0023727715015411377 shape = ()


## 我的观察与实验

1. 现在一次预测需要**两个输入张量** x1 和 x2。x1 和 x2 的 shape 都是 `(6,)`，它们各自的第 i 个元素与 y[i] 属于同一个样本；也就是说 x1、x2 是 6 个样本各自的第一个、第二个特征，样本数是 6，而不是 2。
2. 对照上周：预测公式从 `w*x + b` 变成 `w1*x1 + w2*x2 + b`；参数列表从一个 w 变成 w1、w2 三个参数；损失仍然是先逐样本算误差再 `.mean()`，只是现在预测值由两个输入共同决定。优化器的基本步骤**没有变**：清空梯度 → 预测 → 算损失 → backward → step。
3. 最终参数约为 w1≈2.0、w2≈-3.0、b≈1.0；y_pred 与 y 几乎重合，逐样本绝对误差都在 0.01 量级，平均损失约 0.002。判断训练有效的依据：平均损失从初始的较大值下降到接近 0，预测值 y_pred 与真实 y 的逐样本误差很小，且三个参数收敛到稳定值。
4. 这次 w1=w2=b=0 仍然可以开始学习，是因为预测公式是线性的 `w1*x1 + w2*x2 + b`，损失对每个参数的梯度在零点并不为零（只要 x1、x2 不全为 0，误差就会产生非零梯度），参数可以沿梯度方向更新。这与上周"平方参数"例子不同——平方函数在零点梯度为 0，零初始化会卡住。
